In [1]:
import os;
import sys;
import pickle;
import tensorflow as tf;
from tensorflow.keras import models;

In [2]:
%pwd

'c:\\Users\\chait\\Parkinsons\\research'

In [3]:
os.chdir('../');

In [4]:
from dataclasses import dataclass
from pathlib import Path;

In [ ]:
@dataclass
class ModelTrainConfig:
    data_source: Path;
    model_initial_source: Path;
    model_final_source:Path;

@dataclass
class ModelTrainingParams:
    learning_rate: float;
    epochs: int;
    batch_size: int;

In [6]:
import pandas as pd;

In [24]:
class ModelTrain:
    def __init__(self, model_train_config: ModelTrainConfig):
        self.data_source = model_train_config.data_source;
        self.model_initial_source = model_train_config.model_initial_source;
        self.model_final_source = model_train_config.model_final_source;

    def train(self, model_train_params: ModelTrainingParams):
        model = tf.keras.models.load_model(str(self.model_initial_source));
        X = pd.read_csv(Path(self.data_source) / 'X_train.csv');
        y = pd.read_csv(Path(self.data_source) / 'y_train.csv');
        train_dataset = tf.data.Dataset.from_tensor_slices((X.values, y.values))
        train_dataset = train_dataset.shuffle(buffer_size = len(X)).batch(model_train_params.batch_size)

        print_callback = tf.keras.callbacks.LambdaCallback(
            on_epoch_end=lambda epoch, logs: print(
            f"Epoch {epoch+1}/{model_train_params.epochs} - loss: {logs['loss']:.4f}"
            ) if (epoch + 1) % 100 == 0 else None
        )
        history = model.fit(
            train_dataset, epochs = model_train_params.epochs,
            verbose = 0,
            callbacks = [print_callback]
        )
        os.makedirs(self.model_final_source, exist_ok=True);
        final_path = os.path.join(str(self.model_final_source), "model.keras");
        model.save(final_path);

In [20]:
from exceptions.ModelBuildException import ModelBuildException

In [21]:
import yaml

In [25]:
try:
    with open('config.yaml') as file:
        data = yaml.safe_load(file);
    with open('params.yaml') as file:
        params = yaml.safe_load(file);

    model_train_confi = data['model_training'];
    model_para = params['model_train_params'];

    model_train_config: ModelTrainConfig = ModelTrainConfig(model_train_confi['data_source'], model_train_confi['model_initial_source'], model_train_confi['model_final_source'])
    model_params: ModelTrainingParams = ModelTrainingParams(model_para['learning_rate'], model_para['epochs'], model_para['batch_size']);

    model_train = ModelTrain(model_train_config);
    model_train.train(model_params);
except Exception as e:
    raise ModelBuildException(e, sys);

c:\Users\chait\Parkinsons\venv\lib\site-packages\keras\src\saving\saving_lib.py:868: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 100/1000 - loss: 0.4926
Epoch 200/1000 - loss: 0.4791
Epoch 300/1000 - loss: 0.3732
Epoch 400/1000 - loss: 0.3328
Epoch 500/1000 - loss: 0.2425
Epoch 600/1000 - loss: 0.1991
Epoch 700/1000 - loss: 0.1429
Epoch 800/1000 - loss: 0.1190
Epoch 900/1000 - loss: 0.1328
Epoch 1000/1000 - loss: 0.1528
